In [ ]:
"""
Demo script for piboufilings.
Downloads, parses, and queries SEC filings. Default backend is DuckDB
(single piboufilings.duckdb file under base_dir); set export_format="csv"
for the legacy period-partitioned CSV output.
"""


In [ ]:
from piboufilings import get_filings

# Replace with your actual contact (required by SEC fair-access rules).
USER_NAME = "Your Name or Company"
USER_AGENT_EMAIL = "yourname@example.com"


## 1. Download and parse `13F-HR` + `NPORT-P` filings

The default `export_format="duckdb"` writes a single `piboufilings.duckdb`
under `base_dir`. Install the extra once with `pip install 'piboufilings[duckdb]'`.


In [ ]:
print("=== Example 1: Berkshire Hathaway 13F + NPORT filings ===")

berkshire_cik = "0001067983"  # https://www.sec.gov/search-filings/cik-lookup

get_filings(
    user_name=USER_NAME,
    user_agent_email=USER_AGENT_EMAIL,
    cik=berkshire_cik,
    form_type=["13F-HR", "NPORT-P"],
    start_year=2020,
    end_year=2025,
    base_dir="./my_sec_data",
    log_dir="./my_sec_logs",
    raw_data_dir="./my_sec_raw_data",
    keep_raw_files=True,
    max_workers=5,
    export_format="duckdb",   # default; pass "csv" for the legacy files
)


## 2. Inspect the DuckDB output

All parsed data lives in `./my_sec_data/piboufilings.duckdb`. Each dataset is a
table with the same columns documented in the README.


In [ ]:
import duckdb

db_path = "./my_sec_data/piboufilings.duckdb"
con = duckdb.connect(db_path)

# Top 10 holdings by share value for the most recent 13F period we have.
con.sql("""
    SELECT NAME_OF_ISSUER, SHARE_VALUE, SHARE_AMOUNT
    FROM holdings_13f
    WHERE CONFORMED_DATE = (SELECT MAX(CONFORMED_DATE) FROM holdings_13f)
    ORDER BY SHARE_VALUE DESC
    LIMIT 10
""").df()


In [ ]:
con.sql("""
    SELECT CIK, REPORT_TYPE, COMPANY_NAME, FILED_DATE, TOTAL_VALUE, SEC_FILING_URL
    FROM filing_info_13f
    ORDER BY FILED_DATE DESC
    LIMIT 5
""").df()


## 3. Section 16 (Forms 3/4/5) insider filings

Forms 3/4/5 are routed via the `SECTION-6` alias.


In [ ]:
print("=== Example 2: Recent Form 4 (insider trades) for Apple ===")
insider_cik = "0000320193"  # Apple Inc.
get_filings(
    user_name=USER_NAME,
    user_agent_email=USER_AGENT_EMAIL,
    cik=insider_cik,
    form_type="SECTION-6",
    start_year=2025,
    end_year=2025,
    base_dir="./my_sec_data",
    log_dir="./my_sec_logs",
    raw_data_dir="./my_sec_raw_data",
    keep_raw_files=True,
    max_workers=1,
    export_format="duckdb",
)


In [ ]:
# Aggregate insider sell transactions by reporting owner.
con.sql("""
    SELECT RPT_OWNER_NAME,
           SUM(CAST(TRANSACTION_SHARES AS DOUBLE) *
               CAST(TRANSACTION_PRICE_PER_SHARE AS DOUBLE)) AS dollar_gross
    FROM transactions_sec16
    WHERE TRANSACTION_CODE IN ('S', 'F', 'J')
    GROUP BY 1
    ORDER BY dollar_gross DESC NULLS LAST
    LIMIT 10
""").df()


In [ ]:
con.close()

# CSV fallback: if you prefer period-partitioned CSVs, re-run with
# export_format="csv" and read them directly, e.g.:
#
# import pandas as pd
# from pathlib import Path
# latest = sorted(Path("./my_sec_data").glob("13f_holdings_*.csv"))[-1]
# df = pd.read_csv(latest)
